# Kalshi Sports Market Data for Arbitrage (Demo)

### Install dependencies

Installs the minimal Python packages used in this tutorial (`pandas`, `requests`).

In [ ]:
!pip -q install pandas requests


### Imports + Kalshi REST helper

Sets up KALSHI API used for all requests in this notebook

In [ ]:

import time
import requests
import pandas as pd

KALSHI_BASE = "https://api.elections.kalshi.com/trade-api/v2"

# a small GET helper with retries
def get_json(path, params=None, timeout=20, retries=3):
    """
    Simple GET helper with retries.
    path can be "/markets" or full "https://..." url.
    """
    url = path if path.startswith("http") else f"{KALSHI_BASE}{path}"
    last_err = None
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last_err = e
            time.sleep(1.5 * (attempt + 1))
    raise RuntimeError(f"GET failed after {retries} tries: {url} params={params}\n{last_err}")


### Explore “sports filters” metadata

Fetches `/search/filters_by_sport`, which is **filter metadata**
Use this to discover what sport names / filter fields exist.

In [ ]:
# Sports filters (this is just filter metadata, not markets)

filters = get_json("/search/filters_by_sport")
print("sports ordering (first 20):", (filters.get("sport_ordering") or [])[:20])

# Peek one sport's available filters (edit SPORT to explore)
SPORT = "Football"
by_sport = (filters.get("filters_by_sports") or {})
print(f"\nFilters for {SPORT}:")
print(by_sport.get(SPORT, {}))


sports ordering (first 20): ['All sports', 'Football', 'Basketball', 'Hockey', 'Soccer', 'Tennis', 'Golf', 'MMA', 'Baseball', 'Boxing', 'Chess', 'Cricket', 'Esports', 'Motorsport', 'Olympics']

Filters for Football:
{'competitions': {'College Football': {'scopes': ['Futures', 'Awards', 'Championship', 'Conferences', 'Events']}, 'Pro Football': {'scopes': ['Games', 'Futures', '1st Half Total', '1st Quarter Spread', '1st Quarter Total', '1st Quarter Winner', 'Awards', 'Draft', 'Events']}}, 'scopes': ['Games', 'Futures', '1st Half Total', '1st Quarter Spread', '1st Quarter Total', '1st Quarter Winner', 'Awards', 'Championship', 'Draft', 'Events']}


### Understand `/series`

Kalshi “series” are often **recurring templates**. Depending on how Kalshi categorizes series at the moment

This cell:
- Pulls **all** series
- Prints available categories
- Shows whether any series are tagged as sports
- Displays a small DataFrame preview


In [ ]:
series_all = get_json("/series", params={"include_volume": "true"}).get("series") or []
print("Total series returned:", len(series_all))

# What categories actually exist in series?
cats = sorted({(s.get("category") or "").strip() for s in series_all if s.get("category")})
print("Some series categories:", cats[:30])

sports_series = [s for s in series_all if (s.get("category") or "").lower() == "sports"]
print("Series with category='sports':", len(sports_series))

# If you still want to inspect a few series:
pd.DataFrame(series_all)[["ticker","title","category","volume_fp"]].head(20) if series_all else "No series returned."


Total series returned: 8109
Some series categories: ['Climate and Weather', 'Companies', 'Crypto', 'Economics', 'Education', 'Elections', 'Entertainment', 'Financials', 'Health', 'Mentions', 'Politics', 'Science and Technology', 'Social', 'Sports', 'Transportation', 'World']
Series with category='sports': 1111


,ticker,title,category,volume_fp
0,KXBORDERBILL,Border bill,Politics,852101.00
1,KXATTYGENID,AG race ID,Elections,625.00
2,KXBELGIANPLGAME,Belgian Pro League Game,Sports,5357862.00
3,KXKHMI,Harris wins but loses Michigan,,35622.00
4,KXUKBANGROK,Will the UK ban Grok?,Politics,70188.00
5,KXMLSGAME,Major League Soccer Game,Sports,21677235.00
6,KXCHOPIN,CHOPIN,Entertainment,0.00
7,HOUSENJ5,House NJ's 5th,Politics,1422.00
8,KXMEEKMILLAI,When will Meek Mill launch his AI tool?,Companies,2501.00
9,NEPA,NEPA permitting reform bill,Politics,0.00


###  Fetch a sport markets table

Queries `/markets` and builds a DataFrame with the key fields you’ll likely filter on:

- `yes_bid_dollars`, `yes_ask_dollars` (spread)
- `volume_fp`, `open_interest_fp`
- `last_price_dollars`
- `close_time`, etc.



In [ ]:
# Make a "quality filter" view (liquidity + spread) from your df_kalshi / df_sports_like
import pandas as pd

df = df_kalshi.copy()  # change if your dataframe name is different

# Numeric conversion (Kalshi returns strings sometimes)
for c in ["yes_bid_dollars","yes_ask_dollars","last_price_dollars","volume_fp","open_interest_fp"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# Spread + midpoint
df["yes_spread"] = df["yes_ask_dollars"] - df["yes_bid_dollars"]
df["yes_mid"] = (df["yes_ask_dollars"] + df["yes_bid_dollars"]) / 2

# Tradable-ish thresholds (tune these for you need)
MIN_VOL = 50.0
MIN_OI  = 50.0
MAX_SPREAD = 0.10

good = df[
    ((df["volume_fp"].fillna(0) >= MIN_VOL) | (df["open_interest_fp"].fillna(0) >= MIN_OI)) &
    (df["yes_spread"].fillna(999) <= MAX_SPREAD) &
    (df["yes_bid_dollars"].fillna(0) > 0) &          # avoid “no buyers” markets
    (df["yes_ask_dollars"].notna())
].copy()

# Sort: most liquid first, then tightest spread
good = good.sort_values(["volume_fp","open_interest_fp","yes_spread"], ascending=[False, False, True])

display(good[[
    "ticker","title","status",
    "yes_bid_dollars","yes_ask_dollars","yes_spread",
    "last_price_dollars","volume_fp","open_interest_fp","close_time"
]].head(50))

print("Total markets in df:", len(df))
print("Filtered 'good' markets:", len(good))


,ticker,title,status,yes_bid_dollars,yes_ask_dollars,yes_spread,last_price_dollars,volume_fp,open_interest_fp,close_time
1325,KXNBASPREAD-26JAN25SACDET-DET13,Detroit wins by over 13.5 Points?,active,0.52,0.57,0.05,0.54,115119.0,95258.0,2026-02-08T20:00:00Z
1552,KXATPMATCH-26JAN25SHERUU-SHE,Will Ben Shelton win the Shelton vs Ruud : Rou...,active,0.58,0.59,0.01,0.59,76126.0,73044.0,2026-02-08T23:00:00Z
659,KXATPMATCH-26JAN26ALCDE-ALC,Will Carlos Alcaraz win the Alcaraz vs de Mina...,active,0.80,0.82,0.02,0.82,75231.0,75193.0,2026-02-10T00:30:00Z
1528,KXNBAMENTION-26JAN25DENMEM-INJU,What will the announcers say during Denver at ...,active,0.06,0.07,0.01,0.07,67414.0,49470.0,2026-01-26T01:30:00Z
1553,KXATPMATCH-26JAN25SHERUU-RUU,Will Casper Ruud win the Shelton vs Ruud : Rou...,active,0.41,0.42,0.01,0.41,59366.0,58324.0,2026-02-08T23:00:00Z
1286,KXNBASPREAD-26JAN25DENMEM-MEM2,Memphis wins by over 2.5 Points?,active,0.53,0.61,0.08,0.62,56226.0,38234.0,2026-02-08T20:30:00Z
1306,KXNBATOTAL-26JAN25SACDET-240,Sacramento at Detroit: Total Points,active,0.46,0.50,0.04,0.48,53419.0,37318.0,2026-02-08T20:00:00Z
1326,KXNBASPREAD-26JAN25SACDET-DET10,Detroit wins by over 10.5 Points?,active,0.61,0.67,0.06,0.62,53166.0,46076.0,2026-02-08T20:00:00Z
658,KXATPMATCH-26JAN26ALCDE-DE,Will Alex de Minaur win the Alcaraz vs de Mina...,active,0.18,0.20,0.02,0.20,37882.0,30262.0,2026-02-10T00:30:00Z
1569,KXWTAMATCH-26JAN24INGSWI-ING,Will Maddison Inglis win the Inglis vs Swiatek...,active,0.02,0.04,0.02,0.02,35186.0,34755.0,2026-02-08T00:00:00Z


Total markets in df: 1570
Filtered 'good' markets: 276


### Live feed for specific market

In [ ]:
# Auto-pick an ACTIVE market, show snapshot + orderbook, then poll for 5 minutes (auto-stop), you can change the time based on your need

import time
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
import pandas as pd

ET = ZoneInfo("America/New_York")

def _to_float(x):
    try:
        return float(x)
    except Exception:
        return None

def _get_market_obj(data):
    return data["market"] if isinstance(data, dict) and "market" in data else data

def _get_orderbook_obj(ob_raw):
    return ob_raw.get("orderbook_fp") or ob_raw.get("orderbook") or ob_raw

def pick_active_ticker(df, max_checks=150, sleep_sec=0.12):
    """
    Picks a ticker that looks "live":
    - tradable/open (when fields exist)
    - not the empty book pattern (yes_bid==0 and yes_ask==1)
    - orderbook has at least 1 level on YES or NO
    """
    tickers = df["ticker"].dropna().astype(str).unique().tolist()[:max_checks]

    for t in tickers:
        try:
            snap = get_json(f"/markets/{t}")
            m = _get_market_obj(snap)

            # Skip closed/non-tradable if fields exist
            if m.get("status") in {"closed", "settled", "canceled"}:
                continue
            if m.get("is_tradable") is False:
                continue

            yb = _to_float(m.get("yes_bid_dollars"))
            ya = _to_float(m.get("yes_ask_dollars"))

            # Filter out empty-book pattern (common in inactive markets)
            if yb is not None and ya is not None and (yb == 0.0 and ya == 1.0):
                continue

            ob_raw = get_json(f"/markets/{t}/orderbook")
            ob = _get_orderbook_obj(ob_raw)
            yes = ob.get("yes") or ob.get("yes_dollars") or []
            no  = ob.get("no")  or ob.get("no_dollars")  or []

            if (len(yes) + len(no)) > 0:
                return t

        except Exception:
            pass

        time.sleep(sleep_sec)

    return None

# 1) Auto-pick an active market from `good`
MARKET_TICKER = pick_active_ticker(good, max_checks=200)
if not MARKET_TICKER:
    # Fallback: just pick first one if nothing matched
    MARKET_TICKER = str(good.iloc[0]["ticker"])
    print("⚠️ Could not find an active orderbook quickly; falling back to:", MARKET_TICKER)
else:
    print("✅ Picked active MARKET_TICKER =", MARKET_TICKER)

# 2) Snapshot once
snap = get_json(f"/markets/{MARKET_TICKER}")
m = _get_market_obj(snap)

print("\nTitle:", m.get("title"))
print("YES bid/ask:", m.get("yes_bid_dollars"), "/", m.get("yes_ask_dollars"))
print("NO  bid/ask:", m.get("no_bid_dollars"), "/", m.get("no_ask_dollars"))
print("Last price:", m.get("last_price_dollars"))

# 3) Orderbook once
ob_raw = get_json(f"/markets/{MARKET_TICKER}/orderbook")
ob = _get_orderbook_obj(ob_raw)

yes = ob.get("yes") or ob.get("yes_dollars") or []
no  = ob.get("no")  or ob.get("no_dollars")  or []

print("\nTop YES levels:", yes[:5])
print("Top NO  levels:", no[:5])

# 4) Poll loop (runs for 5 minutes, then stops)
def poll_kalshi_market(ticker, every_sec=2.0, run_minutes=5):
    end_ts = time.time() + run_minutes * 60
    print(f"\nPolling {ticker} every {every_sec}s for ~{run_minutes} minutes...")

    rows = []
    while time.time() < end_ts:
        try:
            data = get_json(f"/markets/{ticker}")
            m = _get_market_obj(data)

            now_utc = datetime.now(timezone.utc)
            now_et = now_utc.astimezone(ET)

            row = {
                "ts_utc": now_utc.isoformat(),
                "ts_est": now_et.isoformat(),
                "yes_bid": m.get("yes_bid_dollars"),
                "yes_ask": m.get("yes_ask_dollars"),
                "last": m.get("last_price_dollars"),
                "volume_fp": m.get("volume_fp"),
                "open_interest_fp": m.get("open_interest_fp"),
            }
            rows.append(row)

            # Print in EST so it looks normal for you
            tlabel = now_et.strftime("%H:%M:%S")
            print(f"[{tlabel} EST] YES {row['yes_bid']}/{row['yes_ask']}  "
                  f"LAST {row['last']}  VOL {row['volume_fp']}  OI {row['open_interest_fp']}")

        except Exception as e:
            # If rate-limited or transient error, just keep going
            print("⚠️ poll error:", str(e)[:120])

        remaining = end_ts - time.time()
        if remaining <= 0:
            break
        time.sleep(min(every_sec, remaining))

    print("\nDone. Collected", len(rows), "rows.")
    return pd.DataFrame(rows)

df_live = poll_kalshi_market(MARKET_TICKER, every_sec=2.0, run_minutes=5)
df_live.tail()


✅ Picked active MARKET_TICKER = KXATPMATCH-26JAN25SHERUU-SHE

Title: Will Ben Shelton win the Shelton vs Ruud : Round Of 16 match?
YES bid/ask: 0.6300 / 0.6400
NO  bid/ask: 0.3600 / 0.3700
Last price: 0.6400

Top YES levels: [['0.0100', '3000.00'], ['0.0200', '3198.00'], ['0.0300', '200.00'], ['0.0400', '300.00'], ['0.0500', '300.00']]
Top NO  levels: [['0.0100', '2587.00'], ['0.0200', '800.00'], ['0.0300', '246.00'], ['0.0400', '335.00'], ['0.0500', '300.00']]

Polling KXATPMATCH-26JAN25SHERUU-SHE every 2.0s for ~5 minutes...
[00:27:09 EST] YES 0.6300/0.6400  LAST 0.6400  VOL 488117.00  OI 482180.00
[00:27:11 EST] YES 0.6300/0.6400  LAST 0.6400  VOL 488117.00  OI 482180.00
[00:27:13 EST] YES 0.6300/0.6400  LAST 0.6400  VOL 488117.00  OI 482180.00
[00:27:15 EST] YES 0.6300/0.6400  LAST 0.6400  VOL 488117.00  OI 482180.00
[00:27:17 EST] YES 0.6300/0.6400  LAST 0.6400  VOL 488117.00  OI 482180.00
[00:27:19 EST] YES 0.6300/0.6400  LAST 0.6400  VOL 488117.00  OI 482180.00
[00:27:21 EST] YE

,ts_utc,ts_est,yes_bid,yes_ask,last,volume_fp,open_interest_fp
142,2026-01-26T05:31:59.907749+00:00,2026-01-26T00:31:59.907749-05:00,0.6200,0.6400,0.6400,501087.00,495150.00
143,2026-01-26T05:32:01.940888+00:00,2026-01-26T00:32:01.940888-05:00,0.6200,0.6400,0.6400,501087.00,495150.00
144,2026-01-26T05:32:03.983733+00:00,2026-01-26T00:32:03.983733-05:00,0.6200,0.6400,0.6400,501087.00,495150.00
145,2026-01-26T05:32:06.020696+00:00,2026-01-26T00:32:06.020696-05:00,0.6200,0.6400,0.6400,501087.00,495150.00
146,2026-01-26T05:32:08.061393+00:00,2026-01-26T00:32:08.061393-05:00,0.6200,0.6400,0.6400,501087.00,495150.00


## More Kalshi API features

This starter kit demonstrates the **core building blocks** you need for the contest (market discovery, basic liquidity/spread filtering, and a simple live polling feed for bid/ask).

If you need **more features beyond the base examples here** (pagination, historical data/candlesticks, order placement, positions, additional endpoints, etc.), please refer to the **official Kalshi API documentation**:

https://docs.kalshi.com/welcome